# Script for Table Preperation of NGS data

## 0. Pre-adjusted settings
### 0.1. Packages

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import os
import re
from Bio.Seq import Seq
from itertools import product
import inspect
import matplotlib as plt


### 0.2. Paths

In [ ]:
#---------------------------------
#---------------------------------
#---------------------------------

#general Path

general_dir = Path('/Users/kollybook/Library/Mobile Documents/com~apple~CloudDocs/Kolja_Hildenbrand/Uni/Master_Infectious_Diseases/Grimm_internship/Report_Grimm/Data') # <----- Hier den Server Path anpassen
os.makedirs(general_dir, exist_ok=True)

#---------------------------------
#---------------------------------
#---------------------------------

# Path for NGS data
NGS_dir = general_dir/'NGS_data'
os.makedirs(NGS_dir, exist_ok=True)

# Path for NGS_tissue
NGS_tissue_dir = NGS_dir/'tissues'
os.makedirs(NGS_tissue_dir, exist_ok=True)

# Path for NGS_input
NGS_input_dir = NGS_dir/'input_lib'
os.makedirs(NGS_input_dir, exist_ok=True)

# Path for tech_rep
NGS_tech_rep_dir = NGS_dir/'tech_rep'
os.makedirs(NGS_tech_rep_dir, exist_ok=True)

# Path for tables
tables_dir = general_dir/'Tables'
os.makedirs(tables_dir, exist_ok=True)

#Path for plots
plots_dir = general_dir/'Figures'
os.makedirs(plots_dir, exist_ok=True)


### 0.3 Definition of functions for processing the tables

#### 0.3.1. extract sample folder names

In [ ]:
def ext_sample_folders(folder_dir):
    folders = sorted([
        f.name 
        for f in folder_dir.iterdir() 
        if not f.name.startswith(".")
    ])
    return folders

#### 0.3.2. Create lists based on sample folders

In [ ]:
def extract_lists (folders):
    tissues = []
    ext_types = []

    for folder in folders:
        parts = folder.split("_")

        ext, tissue = parts

        if ext not in ext_types:
            ext_types.append(ext)

        if tissue not in tissues:
            tissues.append(tissue)

    return tissues, ext_types

#### 0.3.3. Load csv files

In [ ]:
def load_csv_files(path, folders, mouse_ids):
    """
    Load CSV files into nested dict:
    NGS_files[tissue][ext][mouse_id] = dataframe
    """

    path = Path(path)
    NGS_files = {}

    for folder in folders:
        folder_path = path / folder

        parts = folder.split("_")

        ext = parts[0]     # cDNA or gDNA
        tissue = parts[1]  # liver or heart

        NGS_files.setdefault(tissue, {}).setdefault(ext, {})

        # ext-specific prefix
        if ext == "cDNA":
            prefix = "c"
        elif ext == "gDNA":
            prefix = "g"

        # alle csv files im Ordner
        csv_files = [f for f in folder_path.iterdir() if f.is_file() and f.suffix.lower() == ".csv"]

        for mouse_id in mouse_ids:
            matching_files = []

            for file in csv_files:
                name_lower = file.name.lower()

                if name_lower.startswith(prefix) and mouse_id.lower() in name_lower:
                    matching_files.append(file)
                    
            df = pd.read_csv(matching_files[0])
            NGS_files[tissue][ext][mouse_id] = df

    return NGS_files
            

#### 0.3.4. Load input librarys for tissue and extraction_type

In [ ]:
# Load input library
def load_input_library ():
    df = pd.read_csv(NGS_input_dir / 'input_lib_outputg35_input_lib_S35_R1_001_PV.csv')
    input_library = {}
    for tissue in TISSUES:
        input_library[tissue] = {}
        for ext in EXT:
            input_library[tissue][ext] = df.copy()
    return input_library

#### 0.3.5. translate nt --> AA sequence

In [ ]:
def translate_to_AA(files):
    AA_files = {}

    for tissue in TISSUES:
        AA_files[tissue] = {}

        for ext in EXT:
            item = files[tissue][ext]

            # check if file input_library 
            if isinstance(item, pd.DataFrame):
                df = item.copy()
                df["AA_sequence"] = df['Peptide'].apply(lambda x: str(Seq(x).translate()))

                df_aa = (
                    df.groupby("AA_sequence", as_index=False)['Count']
                    .sum()
                    .sort_values('Count', ascending=False)
                    .reset_index(drop=True)
                )

                AA_files[tissue][ext] = df_aa

            # Check if file is sample with Mouse_ID
            elif isinstance(item, dict):
                AA_files[tissue][ext] = {}

                for sample_key, df in item.items():
                    df = df.copy()
                    df["AA_sequence"] = df['Peptide'].apply(lambda x: str(Seq(x).translate()))

                    df_aa = (
                        df.groupby("AA_sequence", as_index=False)['Count']
                        .sum()
                        .sort_values('Count', ascending=False)
                        .reset_index(drop=True)
                    )

                    AA_files[tissue][ext][sample_key] = df_aa

    return AA_files


#### 0.3.6. Remove AA_seq with stop codon

In [ ]:
def remove_stop_codon (files):
    AA_no_stop_files = {}

    for tissue in TISSUES:
        AA_no_stop_files[tissue]= {}

        for ext in EXT:
            item = files[tissue][ext]

            # check if file input_library 
            if isinstance(item, pd.DataFrame):
                df = item.copy()
                df = df[~df["AA_sequence"].str.contains(r"\*")]
                
                AA_no_stop_files[tissue][ext] = df
            # check if dict (Samples)
            elif isinstance(item, dict):
                AA_no_stop_files[tissue][ext] = {}
                
                for sample_key, df in item.items():
                    df = df.copy()
                    df = df[~df["AA_sequence"].str.contains(r"\*")]
                    
                    AA_no_stop_files[tissue][ext][sample_key] = df
                
    return AA_no_stop_files

#### 0.3.7. Merge AA_seq together RENAME

In [ ]:
# merge all mouse ID together and create pseudo AA_seq and frameshift +1 for sample and input_library
def create_long_AA_table (files):
    long_table = {}

    for tissue in TISSUES:
        long_table[tissue]= {}

        for ext in EXT:
            long_table[tissue][ext]= {}
            long = []
            for mouse_ID in MOUSE_ID:
                df = files[tissue][ext][mouse_ID].copy()
                long.append(df)
    
            df = pd.concat(long, ignore_index=True)
    
            df = df['AA_sequence'].unique()

            long_table[tissue][ext] = df
    return long_table

#### 0.3.7. Create Pseudo library and frame shift Merge will function before

In [ ]:
def expand_input_library_with_sample_AA(long_table, input_lib):
    expanded_input = {}

    for tissue in TISSUES:
        expanded_input[tissue] = {}

        for ext in EXT:
            aa_sample = long_table[tissue][ext]

            #input library
            df_input = input_lib[tissue][ext].copy()

            # Outer merge mit allen AA_seq aus Samples
            df_merged = pd.DataFrame({"AA_sequence": aa_sample}).merge(
                df_input,
                on="AA_sequence",
                how="outer"
            )
            
            # wenn Count fehlt -> neu durch Samples hinzugefügt
            df_merged["Pseudo"] = 0
            mask = df_merged["Count"].isna()
            df_merged.loc[mask, "Pseudo"] = 1
            
            df_merged["Count"] = df_merged["Count"].fillna(0)
            
            df_merged = (
                df_merged
                .sort_values("Count", ascending=False)
                .reset_index(drop=True)
            )
            df_merged['Count'] = df_merged['Count']+1
            expanded_input[tissue][ext] = df_merged
            

    return expanded_input

#### 0.3.8. Create Pseudo samples with frameshift

In [ ]:
def merge_samples_with_input(files, input_lib):
    merged_files = {}

    for tissue in TISSUES:
        merged_files[tissue] = {}

        for ext in EXT:
            merged_files[tissue][ext] = {}

            # input library for tissue and ext
            df_input = input_lib[tissue][ext].copy()
            df_input = df_input[['AA_sequence', 'Count', 'Pseudo', 'Proportion']]

            # rename input columns
            rename_dict = {
                "Count": "Count_input",
                "Pseudo": "Pseudo_input",
                "Proportion": "Proportion_input"
            }
            df_input = df_input.rename(columns=rename_dict)

            for mouse_ID in MOUSE_ID:
                df_sample = files[tissue][ext][mouse_ID].copy()

                # merge sample + input
                df_merged = df_sample.merge(
                    df_input,
                    on="AA_sequence",
                    how="outer"
                )

                # NaN = 0
                df_merged["Pseudo"] = 0
                mask = df_merged["Count"].isna()
                df_merged.loc[mask, "Pseudo"] = 1
                df_merged = df_merged.fillna(0)

                global_factor = df_merged["Count"].sum() / df_merged["Count_input"].sum()
                df_merged['Count'] = df_merged['Count'] + global_factor

                merged_files[tissue][ext][mouse_ID] = df_merged

    return merged_files

#### 0.3.9. Calculate proportion

In [ ]:
def calculate_tables (files):
    AA_calc_files = {}

    for tissue in TISSUES:
        AA_calc_files[tissue] = {}

        for ext in EXT:
            item = files[tissue][ext]

            # if input library
            if isinstance(item, pd.DataFrame):
                df = item.copy()

                total_count = df["Count"].sum()
                df["Proportion"] = df["Count"] / total_count
                df["Cum_prop"] = df["Proportion"].cumsum()

                df = df.reset_index(drop=True)

                AA_calc_files[tissue][ext] = df

            # if sample with mouse_ID
            elif isinstance(item, dict):
                AA_calc_files[tissue][ext] = {}

                for sample_key, df in item.items():
                    df = df.copy()

                    total_count = df["Count"].sum()

                    total_count = df["Count"].sum()
                    df["Proportion"] = df["Count"] / total_count
                    df["Cum_prop"] = df["Proportion"].cumsum()
                    df['Log2_enrichment'] = np.log2(df['Proportion']/df['Proportion_input'])

                    AA_calc_files[tissue][ext][sample_key] = df[['AA_sequence', 'Count', 'Proportion', 'Cum_prop', 'Log2_enrichment', 'Pseudo_input', 'Pseudo']].sort_values("Log2_enrichment", ascending=False).reset_index(drop=True)

    return AA_calc_files

#### 0.3.10. Create long table with all samples

In [ ]:
def create_long_table (tables):

    value_columns = ['Count', 'Proportion', 'Cum_prop', 'Log2_enrichment', 'Pseudo_input', 'Pseudo']

    # Create new columns for sample identification 
    df_long = []
    for tissue, ext, mouse_ID in product(TISSUES, EXT, MOUSE_ID):
        df = tables[tissue][ext][mouse_ID].copy()

        df["Sample"] = f"{ext}_{tissue}_{mouse_ID}"
        df["Sex"] = get_sex(mouse_ID)
        df["Mouse_ID"] = mouse_ID
        df["Tissue"] = tissue
        df["Extraction_type"] = ext
        df_long.append(df)
    df_long_table_metadata = pd.concat(df_long, ignore_index=True)

    #Reorganization of columns, first AA_sequence, 2nd Sample type, 3rd values
    df_long_table_metadata = df_long_table_metadata[
        [c for c in df_long_table_metadata.columns if c not in value_columns] + value_columns
    ].copy()
    df_long_table = df_long_table_metadata[['AA_sequence', 'Sample']+ value_columns].copy()
    return df_long_table_metadata, df_long_table

#### 0.3.11. Create pooled table

In [ ]:
def create_pooled_table(long_table_metadata):
    df_long = long_table_metadata.copy()

    df_pooled = (
        df_long
        .groupby(["AA_sequence", "Tissue", "Extraction_type"], as_index=False)
        .agg({
            "Log2_enrichment": "mean",
            "Proportion": "mean",
            "Pseudo": "sum",
            "Pseudo_input": "first"
        })
    )

    df_pooled["n_tissue_samples_present"] = 6 - df_pooled["Pseudo"]
    df_pooled = df_pooled.sort_values("Log2_enrichment", ascending=False).reset_index(drop=True)
    return df_pooled

### 0.4. Definition of helper functions
#### 0.4.1. Save tables

In [ ]:
# Save tables
def save_tables (path, table, name):
    folder_path = path
    os.makedirs(folder_path, exist_ok=True)
    table.to_csv(folder_path / f"{name}.csv", index=False)
    

#### 0.4.2. Save plots

In [ ]:
# save plots
def save_plot(fig, path, name):
    os.makedirs(path, exist_ok=True)
    fig.savefig(os.path.join(path, f"{name}.png"),
                dpi=300,
                bbox_inches="tight")

#### 0.4.3. Check sex of mouse_ID

In [ ]:
def get_sex(mouse_id):
    mouse_id = mouse_id.lower()
    
    if mouse_id.startswith("f"):
        return "female"
    elif mouse_id.startswith("m"):
        return "male"
    else:
        return "unknown"

#### 0.4.4. Check dict NOT USED

In [ ]:
#def for checking dict

def check(dictionary, path=None):
    if path is None:
        path = []

    if isinstance(dictionary, dict):
        for key, value in dictionary.items():
            check(value, path + [key])
    else:
        if hasattr(dictionary, "shape"):
            msg = f"{' - '.join(map(str, path))}: {dictionary.shape} | {len(dictionary.columns)} columns"

            for col in dictionary.columns:
                if col.startswith("Count"):
                    msg += f" | {col} sum = {dictionary[col].sum():,.0f}" 
                    if col == 'Count_tissue':
                        count = len(dictionary[dictionary['Count_tissue'] != 0])
                    if col == 'Count_input_lib':
                        count = len(dictionary[dictionary['Count_input_lib'] != 0])
            msg += f'| variants before = {count}'
            print(msg)
        else:
            print(f"{' - '.join(map(str, path))}: {type(dictionary)}")

### 1. Script

### 1.1. Create lists

In [ ]:
FOLDER_NAMES = ext_sample_folders (NGS_tissue_dir)
TISSUES, EXT = extract_lists(FOLDER_NAMES)
MOUSE_ID = ['f1', 'f2', 'f3', 'm1', 'm2', 'm3']

In [ ]:
FOLDER_NAMES, TISSUES, EXT

### 1.2. Load, translate and remove AA_seq with stop_codon from samples

In [ ]:
%%time
# load csv files
NGS_files = load_csv_files(NGS_tissue_dir, FOLDER_NAMES, MOUSE_ID)

# translate nt files to AA files
dict_AA_samples = translate_to_AA (NGS_files)

# remove AA_seq with stop codons
dict_AA_samples_no_stop = remove_stop_codon (dict_AA_samples)

### 1.3. Load, translate and remove AA_seq with stop_codon from input library

In [ ]:
%%time
# load librarys
dict_input_lib = load_input_library()

# translate nt files to AA files
dict_AA_input = translate_to_AA (dict_input_lib)

# remove AA_seq with stop codons
dict_AA_input_no_stop = remove_stop_codon (dict_AA_input)

#### 1.3.1. Save raw data tables with AA_seq

In [ ]:
# Save all raw sample tables
for tissue, ext, mouse_ID in product (TISSUES, EXT, MOUSE_ID):
    save_tables(tables_dir/tissue, dict_AA_samples_no_stop[tissue][ext][mouse_ID], f'df_{ext}_{tissue}_{mouse_ID}_raw' )

# Save raw library table
save_tables(tables_dir, dict_AA_input_no_stop['liver']['gDNA'], 'df_input_lib_raw')

### 1.4. Create tissue and gDNA/cDNA specific input library with frameshift +1

In [ ]:
# merge all mouse ID together and create pseudo AA_seq and frameshift +1 for sample and input_library
unique_AA_seq = create_long_AA_table (dict_AA_samples_no_stop)
dict_sample_input = expand_input_library_with_sample_AA (unique_AA_seq, dict_AA_input_no_stop)

# calculate prop and cum_prop for input
AA_input_calc = calculate_tables (dict_sample_input)

### 1.5. calculate proportion and log2_enrichment for samples with frameshift + (sum(count_sample)/sum(count_input))

In [ ]:
# Merge samples with calc_input
dict_sample_merged = merge_samples_with_input (dict_AA_samples_no_stop, AA_input_calc)

# calc proportion, cumulative prop and Log2_enrichment for samples
dict_samples = calculate_tables (dict_sample_merged)

### 1.6. Create long table with metadata for ploting 

In [ ]:
%%time
# create long table with metadata for ploting and one condense to save
df_long_table_metadata, df_long_table = create_long_table(dict_samples)

### 1.7. Create pooled table

In [ ]:
%%time
# Create pooled table
df_pooled_tissues = create_pooled_table(df_long_table_metadata)

### 1.8. Save all tables

In [ ]:
%%time
# Save all sample tables
for tissue, ext, mouse_ID in product (TISSUES, EXT, MOUSE_ID):
    save_tables(tables_dir/tissue, dict_samples[tissue][ext][mouse_ID], f'df_{ext}_{tissue}_{mouse_ID}_processed' )

# Save all input librarys tables
for tissue, ext in product (TISSUES, EXT):
    save_tables(tables_dir/tissue, AA_input_calc[tissue][ext], f'df_input_library_for_{ext}_{tissue}' )

# Save long table metadata
save_tables(tables_dir, df_long_table_metadata, 'df_long_table_metadata')

# Save long table
save_tables(tables_dir, df_long_table, 'df_long_table')

# Save pooled table
save_tables(tables_dir, df_pooled_tissues, 'df_pooled_table')


In [ ]:

### Think about plots I want to have in my report
# enrichment distribution
# corr between input library and tissue_specific
# proportion distribution
# log2 pooled distribution
# Sex different